# Knoedler stock books — the New York art market — walkthrough

**One row is** one transaction recorded in the Knoedler gallery stock books.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-datasets/blob/main/datasets/03_getty_knoedler/walkthrough.ipynb)

**This notebook is written to be read, not filled in.** Every cell already works. Run them in order, read the explanation above each one, and look at what comes out.

When you reach the end, go back and change one thing — a column name, a number — and run it again. You cannot break the data: it is a separate file and this notebook only reads it.

---

## Step 0 — Load the data

`pd.read_csv` reads the file into a table. We call the table `df`, which is just a name — it is short for *dataframe*.

In Colab you will be asked to upload `data.csv` first.

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

if not os.path.exists('data.csv'):
    try:
        from google.colab import files
        print('Upload data.csv from your dataset folder')
        files.upload()
    except ImportError:
        raise SystemExit('Place data.csv next to this notebook.')

df = pd.read_csv('data.csv')
print(f'{len(df):,} rows and {df.shape[1]} columns')

## Step 1 — Look at it

`.head()` shows the first five rows. Always look before you calculate: you will often spot a problem here that no statistic would tell you about.

In [ ]:
df.head()

## Step 2 — The absence audit

Which columns are incomplete? `.isna()` marks every empty cell, and `.mean()` turns that into a proportion.

This is the same thing as `=COUNTBLANK` in Excel, and it is worth **30% of the assignment mark**. A column that is mostly empty cannot support a confident claim.

In [ ]:
pd.DataFrame({
    'type': df.dtypes.astype(str),
    'missing_%': (df.isna().mean() * 100).round(1),
    'distinct': df.nunique(),
}).sort_values('missing_%', ascending=False)

## Step 3 — Describe one column

`nationality_1` is a **categorical** column — a label, not a quantity. The useful summary is a count of each value.

`value_counts()` does that. In Excel this is a pivot table with `nationality_1` in Rows and Count as the value.

In [ ]:
counts = df['nationality_1'].value_counts()
print(f'{df["nationality_1"].nunique():,} distinct values')
counts.head(10)

## Step 4 — Compare groups

Now the central move of the whole workshop: split the rows by `nationality_1` and summarise `price_amount` within each group.

Read the code left to right: take `df`, group it by `nationality_1`, look at the `price_amount` column, and compute. This is exactly a pivot table.

We ask for the **median** as well as the mean, because if they differ a lot the data is skewed and the mean is describing something that may not exist.

In [ ]:
summary = df.groupby('nationality_1')['price_amount'].agg(['count', 'mean', 'median'])
summary = summary[summary['count'] >= 5].sort_values('median', ascending=False)
summary.round(2).head(12)

## Step 5 — Could the difference be chance?

Take the two largest groups and ask whether their difference is bigger than we would expect from noise.

The **p-value** answers 'could this be chance?'. **Cohen's d** answers 'is it big?'. They are different questions and you need both — with a large dataset almost anything is 'significant'.

In [ ]:
from scipy import stats

top2 = summary.head(2).index.tolist()
if len(top2) == 2:
    a = df[df['nationality_1'] == top2[0]]['price_amount'].dropna()
    b = df[df['nationality_1'] == top2[1]]['price_amount'].dropna()
    t, p = stats.ttest_ind(a, b, equal_var=False)
    pooled = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    d = (a.mean() - b.mean()) / pooled if pooled else np.nan
    print(f'Comparing {top2[0]!r} (n={len(a)}) with {top2[1]!r} (n={len(b)})')
    print(f'  difference in means : {a.mean() - b.mean():,.2f}')
    print(f'  p-value             : {p:.4g}')
    print(f'  Cohen\'s d           : {d:+.2f}')
    print()
    print('p below 0.05 means the difference is unlikely to be chance.')
    print("d of 0.2 is small, 0.5 medium, 0.8 large.")
else:
    print('Not enough groups to compare.')

## Step 6 — One chart

A sorted bar chart. Sorting matters: it hands the reader the ranking instead of making them work it out.

Note the title states the finding rather than naming the columns.

In [ ]:
top = summary.head(10).sort_values('median')
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh([str(i)[:28] for i in top.index], top['median'], color='#4C8DFF')
ax.set_xlabel('median price_amount')
ax.set_title('Median price_amount by nationality_1 (groups with 5+ rows)')
plt.tight_layout()
plt.show()

## Step 7 — What this does not show

Prices sit in several currencies across a century, so summing or averaging the raw `price_amount` column is meaningless. Either restrict to one currency and one period, or say clearly that you are counting transactions rather than value.

Write your limits section before you write your finding. It is easier, it is worth more marks, and it stops you claiming something the data cannot support.

---

## Now change something

Go back and edit one thing, then run the cell again:

- swap `nationality_1` for a different categorical column
- swap `price_amount` for a different numeric column
- change `head(10)` to `head(20)`
- change `>= 5` to `>= 20` and watch small groups disappear

If you get a red error, read the **last** line — it names the problem. `KeyError` almost always means a column name is spelled differently. Reload the notebook and you are back where you started.